# 1 - HCP fMRI preprocessing

This notebook preprocesses HCP resting-state fMRI data for local use.

It assumes the following folder structure:

Data/
└──HCP_task_data
   └──Schafer400_Tian50

Projects/
└── BrainStim/
    ├── src/
    └── notebooks/

So:
- the notebook is inside `Projects/BrainStim/notebooks`
- the source code is inside `Projects/BrainStim/src`
- the raw HCP data is inside `Data/HCP_task_data/Schafer400_Tian50`
- the processed outputs will be saved in `Projects/BrainStim/results/processed`

## What this notebook does

For each selected subject, it:

1. Loads the four HCP resting-state runs
2. Fixes matrix orientation if needed
3. Removes the first time points of each run
4. Applies temporal band-pass filtering
5. Concatenates the runs
6. Builds supervised learning samples using a sliding window
7. Saves the outputs as `.npy` files

## Saved outputs

For each subject, three arrays are saved:

### 1. `signals`
Shape: `(T, N)`

- `T`: total concatenated time points after preprocessing
- `N`: number of brain regions

This contains the filtered fMRI regional signals.

### 2. `inputs`
Shape: `(T - S, N * S)`

- `S`: number of past steps used as input

Each row contains a flattened sliding window of the previous `S` time steps across all regions.

### 3. `targets`
Shape: `(T - S, N)`

Each row contains the signal at the next time step corresponding to the input window.

In [1]:
from pathlib import Path
import sys
import gc
import h5py
import numpy as np
import matplotlib.pyplot as plt
import os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE" 
#To avoid problems with MKL/OpenMP conflicts for importing torch is sometimes needed
#(normally importing torch before numpy could also works)

# -----------------------------------------------------------------------------
# Local path configuration
# -----------------------------------------------------------------------------
# Expected structure:
#
# Data/
# └── HCP_task_data/
#    └── Schafer400_Tian50/
#
# Projects/
# └── BrainStim/
#     ├── src/
#     └── notebooks/
#     └── results/processed
#
# Notebook is assumed to run from:
# Projects/BrainStim/notebooks/

repo_dir = Path.cwd().resolve().parent # BrainStim/
sys.path.insert(0, str(repo_dir)) #To import the src codes
root_dir = repo_dir.parent.parent


src_dir = repo_dir / "src"
data_dir = root_dir / "Data" / "HCP_task_data" / "Schafer400_Tian50"
save_dir = repo_dir / "Results"/ "processed"

print("Root directory      :", root_dir)
print("Repository directory:", repo_dir)
print("Source directory    :", src_dir)
print("Data directory      :", data_dir)
print("Save directory      :", save_dir)

from src.preprocessing_hcp import bandpass_filter_timeseries
from src.NPI import multi2one

Root directory      : C:\Users\tomas\Documents\PhD
Repository directory: C:\Users\tomas\Documents\PhD\Projects\BrainStim
Source directory    : C:\Users\tomas\Documents\PhD\Projects\BrainStim\src
Data directory      : C:\Users\tomas\Documents\PhD\Data\HCP_task_data\Schafer400_Tian50
Save directory      : C:\Users\tomas\Documents\PhD\Projects\BrainStim\Results\processed


In [2]:
# =============================================================================
# Processing parameters
# =============================================================================
n_nodes = 450               # number of brain regions/parcels to keep
remove_points = 30          # number of initial TRs removed from each run
using_steps = 3             # sliding window length
number_of_subjects = 100     # increase as needed (996 is all)
dtype = np.float32

In [3]:
# =============================================================================
# Locate HCP fMRI run files
# =============================================================================
run_files = {
    "REST1_LR": data_dir / "Schaefer2018_400Parcels_7Networks_order_Tian_Subcortex_S3_REST1_LR.mat",
    "REST1_RL": data_dir / "Schaefer2018_400Parcels_7Networks_order_Tian_Subcortex_S3_REST1_RL.mat",
    "REST2_LR": data_dir / "Schaefer2018_400Parcels_7Networks_order_Tian_Subcortex_S3_REST2_LR.mat",
    "REST2_RL": data_dir / "Schaefer2018_400Parcels_7Networks_order_Tian_Subcortex_S3_REST2_RL.mat",
}

run_order = ["REST1_LR", "REST1_RL", "REST2_LR", "REST2_RL"]

print(f" Found {len(run_files)} fMRI runs:")
for k in run_order:
    print("  -", k)

 Found 4 fMRI runs:
  - REST1_LR
  - REST1_RL
  - REST2_LR
  - REST2_RL


In [4]:
# =============================================================================
# Find subjects present in all runs
# =============================================================================
def list_subjects(h5path, run_key):
    """
    Return sorted subject identifiers present in a given run file.
    """
    with h5py.File(h5path, "r") as f:
        return sorted(f["HCP"][run_key].keys(), key=lambda k: int(k.split("_")[-1]))

subject_sets = [set(list_subjects(run_files[run_key], run_key)) for run_key in run_order]
subject_ids = sorted(set.intersection(*subject_sets), key=lambda k: int(k.split("_")[-1]))

print(f"Subjects present in all runs: {len(subject_ids)}")
subject_ids = subject_ids[:number_of_subjects]
print(f"Subjects selected for processing (ordered): {len(subject_ids)}")

Subjects present in all runs: 996
Subjects selected for processing (ordered): 100


In [5]:
# =============================================================================
# Main preprocessing loop
# =============================================================================

save_dir.mkdir(parents=True, exist_ok=True)

for sid in subject_ids:
    print(f"\nProcessing subject {sid}")
    subj_runs = []

    for run_key in run_order:
        with h5py.File(run_files[run_key], "r") as f:
            ts = f["HCP"][run_key][sid]["ts"][()]
        #print(f"      Original shape for {run_key}: {ts.shape}", end="")
        
        # Ensure shape is (time, regions)
        if ts.shape[0] < ts.shape[1]:
            ts = ts.T
            #print(f" -> transposed to {ts.shape}")
        else:
            #print(" -> kept as is")
            pass

        # Remove initial TRs and keep selected nodes
        ts = ts[remove_points:, :n_nodes].astype(dtype, copy=False)

        # Apply temporal filtering
        ts_filt = bandpass_filter_timeseries(ts).astype(dtype, copy=False)
        subj_runs.append(ts_filt)

        del ts, ts_filt
        gc.collect()

    # Concatenate all runs for the current subject
    signals = np.concatenate(subj_runs, axis=0).astype(dtype, copy=False)

    # Build supervised samples (inputs & targets)
    inputs, targets = multi2one(signals, steps=using_steps)

    #print(f"  signals shape: {signals.shape}")
    #print(f"  inputs  shape: {inputs.shape}")
    #print(f"  targets shape: {targets.shape}")

    # Save outputs
    np.save(save_dir / f"{sid}_signals.npy", signals)
    np.save(save_dir / f"{sid}_inputs.npy", inputs)
    np.save(save_dir / f"{sid}_targets.npy", targets)

    del subj_runs, signals, inputs, targets
    gc.collect()

print("\nProcessing complete.")


Processing subject id_100206

Processing subject id_100307

Processing subject id_100408

Processing subject id_101006

Processing subject id_101107

Processing subject id_101309

Processing subject id_101915

Processing subject id_102008

Processing subject id_102109

Processing subject id_102311

Processing subject id_102513

Processing subject id_102614

Processing subject id_102715

Processing subject id_102816

Processing subject id_103010

Processing subject id_103111

Processing subject id_103212

Processing subject id_103414

Processing subject id_103515

Processing subject id_103818

Processing subject id_104012

Processing subject id_104416

Processing subject id_104820

Processing subject id_105014

Processing subject id_105115

Processing subject id_105216

Processing subject id_105620

Processing subject id_105923

Processing subject id_106319

Processing subject id_106521

Processing subject id_106824

Processing subject id_107018

Processing subject id_107321

Processin